### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_discussion import QuinexDiscussion
from quinex_application.quinex_visualisation import QuinexVisualisation

In [2]:
discussion = QuinexDiscussion("../data/visualisation_files/regex/entity_patterns.json", "../data/visualisation_files/regex/property_patterns.json")
visualisation = QuinexVisualisation()

### **Load Results**

In [3]:
df_quinex_metadata = discussion.load_df("../data/quinex/df_quinex_metadata.parquet")
df_quinex = discussion.load_df("../data/quinex/df_quinex.parquet")

📂 Loaded DataFrame from '../data/quinex/df_quinex_metadata.parquet' with 31633 rows
📂 Loaded DataFrame from '../data/quinex/df_quinex.parquet' with 3244165 rows


In [ ]:
df_quinex["property"].value_counts().iloc[50:100]

### **Filter data**

In [4]:
df = df_quinex.copy()
entity_regex_map = discussion.entity_regex_map
property_regex_map = discussion.property_regex_map

In [ ]:
property_regex_map = {
    "Temperature": property_regex_map["Temperature"]}
property_regex_map

In [ ]:
entity_regex_map = {"PV Plant": entity_regex_map["PV Plant"], "Wind Farm": entity_regex_map["Wind Farm"], "Solar Power Plant": entity_regex_map["Solar Power Plant"]}
entity_regex_map

In [5]:
df["entity_mapping"] = df["entity"].apply(lambda x: discussion.map_entity(x, entity_regex_map))
label_to_base = discussion.create_label_to_base(entity_regex_map)
df["entity_base_mapping"] = df["entity_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["entity_mapping"].notna()]

In [6]:
df["property_mapping"] = df["property"].apply(lambda x: discussion.map_entity(x, property_regex_map))
label_to_base = discussion.create_label_to_base(property_regex_map)
df["property_base_mapping"] = df["property_mapping"].apply(lambda x: discussion.map_base_from_mapping(x, label_to_base))
df = df[df["property_mapping"].notna()]

In [7]:
df["property_mapping"].value_counts()

property_mapping
Installed Capacity       15757
Cost                     13302
Efficiency               11601
Levelized Cost            6690
Lifetime                  4462
CO2 emission              3769
Load / Peak Load          3168
Capital Cost              2199
Capacity Factor           1774
Wind Speed                1634
NPC                       1597
Emission                  1485
GHG emission               941
Operating Temperature      904
Operating Cost             877
Production Cost            844
NPV                        695
LPSP                       641
Electricity Cost           439
Power Density              370
Name: count, dtype: int64

In [ ]:
df["entity_base_mapping"].value_counts()

In [8]:
df[df["property_mapping"] == "Operating Temperature"]["property"].value_counts()

property
operating temperature                            671
nominal operating cell temperature                36
operating temperatures                            35
operating temperature range                       14
nominal operating temperature                     13
                                                ... 
standard operating temperatures                    1
operating ambient temperature                      1
nominal operating cell temperature (Tc, NOCT)      1
representative operating temperature               1
operating temperature (T h )                       1
Name: count, Length: 85, dtype: Int64

In [9]:
df.to_parquet("../data/visualisation_files/df_technologies.parquet")

### **Filter results**

In [ ]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_capacity.parquet")

In [ ]:
technologies_df["property_mapping"].value_counts()

In [ ]:
technologies_df[technologies_df["property_mapping"] == "Capital Cost"]["property"].value_counts().head(50)

In [ ]:
technologies_df[(technologies_df["property_mapping"] == "CO2 emission") & (technologies_df["entity"] == "Berlin")].head(2)

### **Postprocess Units**

In [26]:
technologies_df = pd.read_parquet("../data/visualisation_files/df_technologies.parquet")
technologies_df = technologies_df[technologies_df["property_mapping"] == "Capacity Factor"]
def dicts_to_tuples(units):
    return [(u.get('text', ''), u.get('exponent', None), u.get('unit_uri', ''), u.get('year', None)) 
            for u in units]

technologies_df['unit_normalized'] = technologies_df['unit_normalized'].apply(dicts_to_tuples)

In [19]:
technologies_df["entity_base_mapping"].value_counts()

entity_base_mapping
Wind                    980
PV                      417
Electrolyzer            168
Coal                     78
Hydrogen                 58
Natural Gas              32
Battery                  29
Hybrid Energy System     12
Name: count, dtype: int64

In [24]:
print(technologies_df["unit_normalized"].value_counts())
print("--------------------------------------------------------------------------------")
exclude_values = [
    [(None, None, 'http://qudt.org/vocab/unit/UNITLESS', None)],
    [('%', 1.0, 'http://qudt.org/vocab/unit/PERCENT', None)], 
]
technologies_df = technologies_df[~technologies_df["unit_normalized"].isin(exclude_values)]
print(technologies_df["unit_normalized"].value_counts().head(10))

unit_normalized
[(%, 1.0, http://qudt.org/vocab/unit/PERCENT, None)]                                                            1342
[(None, None, http://qudt.org/vocab/unit/UNITLESS, None)]                                                        384
[(kW, 1.0, http://qudt.org/vocab/unit/KiloW, None)]                                                                7
[(percent, 1.0, http://qudt.org/vocab/unit/PERCENT, None)]                                                         7
[(GW, 1.0, http://qudt.org/vocab/unit/GigaW, None)]                                                                5
[(MW, 1.0, http://qudt.org/vocab/unit/MegaW, None)]                                                                4
[(h, 1.0, http://qudt.org/vocab/unit/HR, None), (yr, -1.0, http://qudt.org/vocab/unit/YR, None)]                   3
[(€, 1.0, http://qudt.org/vocab/currency/EUR, None), (MWh, -1.0, http://qudt.org/vocab/unit/MegaW-HR, None)]       2
[(h, 1.0, http://qudt.org/vocab/unit/HR, None), 

In [27]:
target_unit = [('%', 1.0, 'http://qudt.org/vocab/unit/PERCENT', None)]
# target_unit = [("$", 1.0, "http://qudt.org/vocab/currency/USD", None), ("kWh", -1.0, "http://qudt.org/vocab/unit/KiloW-HR", None)]
technologies_df['value_converted'] = technologies_df.apply(
   lambda row: visualisation.convert_row(row, target_unit), axis=1
   )

In [28]:
technologies_df["unit_normalized"].value_counts()

unit_normalized
[(%, 1.0, http://qudt.org/vocab/unit/PERCENT, None)]                                                            1342
[(None, None, http://qudt.org/vocab/unit/UNITLESS, None)]                                                        384
[(kW, 1.0, http://qudt.org/vocab/unit/KiloW, None)]                                                                7
[(percent, 1.0, http://qudt.org/vocab/unit/PERCENT, None)]                                                         7
[(GW, 1.0, http://qudt.org/vocab/unit/GigaW, None)]                                                                5
[(MW, 1.0, http://qudt.org/vocab/unit/MegaW, None)]                                                                4
[(h, 1.0, http://qudt.org/vocab/unit/HR, None), (yr, -1.0, http://qudt.org/vocab/unit/YR, None)]                   3
[(€, 1.0, http://qudt.org/vocab/currency/EUR, None), (MWh, -1.0, http://qudt.org/vocab/unit/MegaW-HR, None)]       2
[(h, 1.0, http://qudt.org/vocab/unit/HR, None), 

In [17]:
technologies_df = technologies_df[technologies_df["value_converted"].notna()]
technologies_df = technologies_df.reset_index(drop=True)
technologies_df.to_excel("../data/visualisation_files/unit_converted/df_technologies_npv.xlsx")